# Arcadia start

Notebook for wiring `drone_heatmap` + `3d reconstruction` together.

In [1]:
%pip install -e "C:/Users/jletobar3/Projects/drone_heatmap"
%pip install -e "C:/Users/jletobar3/Projects/3d reconstruction"

Obtaining file:///C:/Users/jletobar3/Projects/drone_heatmap
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for priority_map (pyproject.toml): started
  Building editable for priority_map (pyproject.toml): finished with status 'done'
  Created wheel for priority_map: filename=priority_map-0.1.0-0.editable-py3-none-any.whl size=2030 sha256=3c7ded1502b7c4f988c818a91aaee16e6b95eb5ee9eaa4002c385dc4e5512933
  Stored in directory: C:\Users\jletobar3\AppData\Local\Temp\pip-ephem-wheel-cache-g8f1oqjx\wheels\ae\b6\82\6e9b1093c5cd94e11f

In [2]:
from pathlib import Path

from priority_map import run_priority_map
from colmap_reconstruction import ReconstructionResult, reconstruct, project_heatmaps

print("imports ok")

imports ok


In [3]:
image_frames = Path(r"D:\dronevid2")
arcadia_out = Path(r"C:/Users/jletobar3/Projects/arcadia/out")

priority_out = arcadia_out / "priority_map"
colmap_out = arcadia_out / "colmap"

priority_out.mkdir(parents=True, exist_ok=True)
colmap_out.mkdir(parents=True, exist_ok=True)

In [5]:
# 1) Make heatmaps from drone images.
if image_frames.exists():
    priority_result = run_priority_map(
        image_folder=image_frames,
        output_dir=priority_out,
        task="Find cars",
        sam_step=30,
        show=False,
        record=True,
        sam_model_path=Path(r"C:/Users/jletobar3/Projects/drone_heatmap/models/sam3.pt")
    )
    print(priority_result)
else:
    print(f"Set image_frames first: {image_frames}")


GPT Vision inference time: 1.26 seconds
{"forest":{"prompt":"forest","score":15},"field":{"prompt":"field","score":10}}

Ultralytics 8.4.56  Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]
0: 644x644 1 forest, 5 fields, 7602.1ms
Speed: 43.0ms preprocess, 7602.1ms inference, 34.8ms postprocess per image at shape (1, 3, 644, 644)
Recording video to C:\Users\jletobar3\Projects\arcadia\out\priority_map\heatmap.avi using MJPG
Semantic clustering:
labels used for rural area: ['field', 'forest']
----------------------
Recording video to C:\Users\jletobar3\Projects\arcadia\out\priority_map\video.avi using MJPG

GPT Vision inference time: 1.58 seconds
{"forest":{"prompt":"forest","score":15},"field":{"prompt":"field","score":10},"building":{"prompt":"rooftops","score":35}}

WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]
0: 644x644 5 forests, 11 fields, 8 roo

In [ ]:
# 2) Reconstruct the same images with COLMAP.
if image_frames.exists():
    reconstruction = reconstruct(
        input_path=image_frames,
        output_dir=colmap_out,
        skip_frames=0,
        max_image_size=640,
    )
    print(reconstruction.mesh_path)
else:
    print(f"Set image_frames first: {image_frames}")

Found 238 image frame(s)
Staging input images


Staging images: 100%|██████████| 238/238 [00:45<00:00,  5.28it/s]


Prepared 238 image(s) for reconstruction

--- Starting sparse reconstruction ---
Extracting features...
Matching features...
Running sparse reconstruction...


In [ ]:
# 3) Project priority-map heatmaps onto an existing reconstruction.
heatmap_dir = priority_out / "heatmap_imgs"

if colmap_out.exists() and heatmap_dir.exists():
    reconstruction = ReconstructionResult.from_output_dir(colmap_out)
    heatmapped = project_heatmaps(reconstruction, heatmap_dir)
    print(heatmapped.output_mesh_path)
else:
    print("Need colmap_out and priority_out/heatmap_imgs first")